# Pandas — Data Wrangling Made Human

Pandas gives you **labeled, tabular data** with powerful tools to clean, transform, and analyze it.  
Think of it as Excel on steroids — but scriptable and reproducible.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
print(f'Pandas version: {pd.__version__}')

---
## Series & DataFrame

**Series** = labeled 1D array (one column).  
**DataFrame** = labeled 2D table (rows × columns).  
Everything in Pandas is built on these two.

In [ ]:
grades = pd.Series([88, 92, 75, 95, 83], index=['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'])
print(f'Series:\n{grades}\n')
print(f'Mean: {grades.mean():.1f}  |  Max: {grades.max()} ({grades.idxmax()})')

rng = np.random.default_rng(42)
n = 50
df = pd.DataFrame({
    'student_id': range(1001, 1001 + n),
    'name': [f'Student_{i}' for i in range(n)],
    'department': rng.choice(['CS', 'Math', 'Physics', 'Biology'], n),
    'year': rng.choice([1, 2, 3, 4], n),
    'gpa': np.round(rng.uniform(2.0, 4.0, n), 2),
    'hours_studied': np.round(rng.uniform(5, 40, n), 1),
    'exam_score': np.round(rng.normal(72, 12, n).clip(0, 100), 1)
})

mask = rng.random(n) < 0.1
df.loc[mask, 'gpa'] = np.nan
mask2 = rng.random(n) < 0.08
df.loc[mask2, 'exam_score'] = np.nan

df.head(8)

---
## Exploring Data

First thing you do with any dataset: **look at it**.

In [ ]:
print(f'Shape: {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)\n')
print(df.dtypes)
print(f'\n--- Missing values ---')
print(df.isna().sum())
print(f'\n--- Department distribution ---')
print(df['department'].value_counts())

In [ ]:
df.describe().round(2)

---
## Selecting Data — `[]`, `loc`, `iloc`, Boolean Filters

- **`[]`** — columns by name, rows by boolean
- **`loc`** — selection by **labels** (inclusive on both ends)
- **`iloc`** — selection by **integer position** (exclusive on end)
- **`query`** — SQL-like string filtering

In [ ]:
print('--- Single column (Series) ---')
print(df['gpa'].head())

print('\n--- Multiple columns (DataFrame) ---')
print(df[['name', 'department', 'gpa']].head())

print('\n--- loc: rows 0-2, specific columns ---')
print(df.loc[0:2, ['name', 'gpa', 'exam_score']])

print('\n--- iloc: first 3 rows, columns 1-3 ---')
print(df.iloc[:3, 1:4])

In [ ]:
honor_roll = df[df['gpa'] > 3.5]
print(f'Honor roll (GPA > 3.5): {len(honor_roll)} students')

cs_seniors = df[(df['department'] == 'CS') & (df['year'] == 4)]
print(f'CS Seniors: {len(cs_seniors)}')
print(cs_seniors[['name', 'gpa', 'exam_score']])

top_students = df.query('gpa > 3.5 and hours_studied > 25')
print(f'\nHigh GPA + High effort: {len(top_students)}')

---
## Handling Missing Data

Real data is **messy**. You need a strategy for missing values.

In [ ]:
print(f'Missing values before:\n{df.isna().sum()}\n')

print('--- Wrong way: just drop everything ---')
df_dropped = df.dropna()
print(f'Rows after dropna: {len(df_dropped)} (lost {len(df) - len(df_dropped)} rows)\n')

print('--- Right way: impute with column means ---')
df_filled = df.copy()
df_filled['gpa'] = df_filled['gpa'].fillna(df_filled['gpa'].mean())
df_filled['exam_score'] = df_filled['exam_score'].fillna(df_filled['exam_score'].median())
print(f'Missing after fillna:\n{df_filled.isna().sum()}')
print(f'Rows preserved: {len(df_filled)}')

---
## Data Transformation — `apply`, `map`, `replace`, `astype`

Transform columns with functions. `apply` is flexible, `map` is for element-wise.

In [ ]:
df_t = df_filled.copy()

def letter_grade(gpa):
    if gpa >= 3.7: return 'A'
    elif gpa >= 3.3: return 'A-'
    elif gpa >= 3.0: return 'B+'
    elif gpa >= 2.7: return 'B'
    else: return 'C'

df_t['letter_grade'] = df_t['gpa'].apply(letter_grade)
print(df_t[['name', 'gpa', 'letter_grade']].head(8))

df_t['passed'] = df_t['exam_score'].apply(lambda x: x >= 60)
print(f'\nPass rate: {df_t["passed"].mean():.1%}')

dept_map = {'CS': 'Computer Science', 'Math': 'Mathematics', 'Physics': 'Physics', 'Biology': 'Biology'}
df_t['dept_full'] = df_t['department'].map(dept_map)
print(f'\n{df_t[["department", "dept_full"]].drop_duplicates()}')

---
## Groupby & Aggregation — Split-Apply-Combine

The most powerful pattern in Pandas:  
1. **Split** data into groups  
2. **Apply** a function to each group  
3. **Combine** results back together

In [ ]:
dept_stats = df_filled.groupby('department').agg(
    avg_gpa=('gpa', 'mean'),
    avg_exam=('exam_score', 'mean'),
    count=('student_id', 'count'),
    top_gpa=('gpa', 'max'),
    avg_hours=('hours_studied', 'mean')
).round(2)

print('Department Statistics:')
print(dept_stats)

In [ ]:
df_g = df_filled.copy()
df_g['dept_avg_gpa'] = df_g.groupby('department')['gpa'].transform('mean').round(2)
df_g['above_dept_avg'] = df_g['gpa'] > df_g['dept_avg_gpa']
print(df_g[['name', 'department', 'gpa', 'dept_avg_gpa', 'above_dept_avg']].head(10))

print('\n--- Multi-level groupby ---')
multi = df_filled.groupby(['department', 'year'])['gpa'].mean().round(2)
print(multi.head(12))

---
## Merging & Joining

Combine DataFrames like SQL JOINs. Master the four join types.

In [ ]:
students = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana']
})

enrollments = pd.DataFrame({
    'student_id': [1, 1, 2, 2, 3, 5],
    'course': ['Math', 'Physics', 'Math', 'CS', 'Biology', 'CS']
})

print('Students:')
print(students)
print('\nEnrollments:')
print(enrollments)

inner = pd.merge(students, enrollments, left_on='id', right_on='student_id', how='inner')
print(f'\n--- INNER JOIN (only matches) ---\n{inner}')

left = pd.merge(students, enrollments, left_on='id', right_on='student_id', how='left')
print(f'\n--- LEFT JOIN (all students, matching enrollments) ---\n{left}')

outer = pd.merge(students, enrollments, left_on='id', right_on='student_id', how='outer')
print(f'\n--- OUTER JOIN (everything) ---\n{outer}')

In [ ]:
df_q1 = pd.DataFrame({'product': ['A', 'B', 'C'], 'q1_sales': [100, 200, 150]})
df_q2 = pd.DataFrame({'product': ['A', 'B', 'C'], 'q2_sales': [120, 180, 170]})

print('--- Merge on shared column ---')
print(pd.merge(df_q1, df_q2, on='product'))

print('\n--- Concat (stacking rows) ---')
batch1 = pd.DataFrame({'name': ['Alice', 'Bob'], 'score': [88, 92]})
batch2 = pd.DataFrame({'name': ['Charlie', 'Diana'], 'score': [75, 95]})
print(pd.concat([batch1, batch2], ignore_index=True))

---
## Pivot & Melt — Reshaping Data

**Pivot**: long → wide (one row per entity, columns for categories).  
**Melt**: wide → long (unpivot, useful for plotting).

In [ ]:
long_data = pd.DataFrame({
    'student': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob', 'Bob'],
    'subject': ['Math', 'Physics', 'CS', 'Math', 'Physics', 'CS'],
    'score': [92, 88, 95, 78, 85, 90]
})
print('Long format:')
print(long_data)

wide_data = long_data.pivot(index='student', columns='subject', values='score')
print(f'\nPivoted (wide):\n{wide_data}')

melted = wide_data.reset_index().melt(id_vars='student', var_name='subject', value_name='score')
print(f'\nMelted back (long):\n{melted}')

---
## String Operations — `.str` Accessor

Vectorized string methods — no loops needed.

In [ ]:
emails = pd.Series([
    'alice.smith@university.edu',
    'BOB.Jones@company.com',
    'charlie_brown@school.org',
    'DIANA.LEE@university.edu'
])

print(f'Lowercase: {emails.str.lower().tolist()}')
print(f'Contains "university": {emails.str.contains("university").tolist()}')
print(f'Domains: {emails.str.split("@").str[1].tolist()}')
print(f'Name part: {emails.str.split("@").str[0].str.replace(".", " ").str.title().tolist()}')

---
## DateTime Operations

Parse dates, extract components, resample time series.

In [ ]:
dates = pd.date_range('2024-01-01', periods=365, freq='D')
rng = np.random.default_rng(42)
daily_revenue = pd.DataFrame({
    'date': dates,
    'revenue': np.round(rng.normal(10000, 2000, 365).clip(2000, 20000), 2)
})
daily_revenue['date'] = pd.to_datetime(daily_revenue['date'])
daily_revenue.set_index('date', inplace=True)

print(daily_revenue.head())

print(f'\nYear:  {daily_revenue.index.year[:3].tolist()}')
print(f'Month: {daily_revenue.index.month[:3].tolist()}')
print(f'Day name: {daily_revenue.index.day_name()[:3].tolist()}')

monthly = daily_revenue.resample('ME').agg(
    total_revenue=('revenue', 'sum'),
    avg_daily=('revenue', 'mean'),
    best_day=('revenue', 'max')
).round(2)
print(f'\nMonthly summary:\n{monthly.head()}')

---
## Sorting & Ranking

Order your data, find the top/bottom N, assign ranks.

In [ ]:
df_s = df_filled.copy()

print('--- Top 5 by GPA ---')
print(df_s.nlargest(5, 'gpa')[['name', 'department', 'gpa']])

print('\n--- Bottom 3 by exam score ---')
print(df_s.nsmallest(3, 'exam_score')[['name', 'exam_score']])

df_s['gpa_rank'] = df_s['gpa'].rank(ascending=False, method='dense').astype(int)
print(f'\n--- With ranks ---')
print(df_s.sort_values('gpa_rank')[['name', 'gpa', 'gpa_rank']].head(8))

---
## Feature Engineering Preview

Creating new features from existing data — a taste of ML preprocessing.

In [ ]:
df_fe = df_filled.copy()

df_fe['study_efficiency'] = (df_fe['exam_score'] / df_fe['hours_studied']).round(2)

df_fe['gpa_bin'] = pd.cut(df_fe['gpa'], bins=[0, 2.5, 3.0, 3.5, 4.0],
                           labels=['Low', 'Medium', 'High', 'Excellent'])
print('Binned GPA distribution:')
print(df_fe['gpa_bin'].value_counts().sort_index())

dummies = pd.get_dummies(df_fe['department'], prefix='dept')
print(f'\nOne-hot encoded departments:\n{dummies.head()}')

df_final = pd.concat([df_fe[['name', 'gpa', 'exam_score', 'study_efficiency']], dummies], axis=1)
print(f'\nFinal feature matrix:\n{df_final.head()}')

---
## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Selection** | `loc` = labels, `iloc` = integers, `[]` = columns/boolean |
| **Missing data** | Don't just drop — impute with mean/median/mode |
| **Groupby** | Split-Apply-Combine — `groupby().agg()` |
| **Merging** | Know your JOINs: inner, left, right, outer |
| **Pivot/Melt** | Reshape for analysis (pivot) or plotting (melt) |
| **Strings** | `.str` accessor — no loops for text |
| **Dates** | `pd.to_datetime` + `.resample()` |
| **Feature eng** | `cut` for bins, `get_dummies` for encoding |